<a href="https://colab.research.google.com/github/SSStarain/agent-learn/blob/aoaoder/gaussian_splatting_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import os
import sys

# --- 步骤 1: 检查环境 ---
# 必须在第一步执行，确保 Colab 分配了 GPU
print("--- 1. 检查环境 ---")
if not torch.cuda.is_available():
    print("错误：未检测到 GPU。")
    print("请点击 Colab 菜单栏的 '运行时' -> '更改运行时类型' -> '硬件加速器' -> 选择 'T4 GPU'。")
    print("然后在 '运行时' -> '断开连接并删除运行时' 后，再重新运行此单元格。")
    # 停止执行，防止后续出错
    sys.exit("GPU not found. Halting execution.")
else:
    print(f"GPU 检测成功: {torch.cuda.get_device_name(0)}")
    print(f"PyTorch CUDA 版本: {torch.version.cuda}")
    print("环境检查通过。")

# --- 步骤 2: 更新构建工具 ---
print("\n--- 2. 更新 pip 和构建工具 ---")
# 加上 -q 来避免过多输出
!pip install --upgrade pip setuptools wheel -q
print("构建工具更新完毕。")

# --- 步骤 3: 克隆仓库和安装 Python 依赖 ---
print("\n--- 3. 克隆仓库并安装 plyfile ---")
%cd /content
if not os.path.exists('gaussian-splatting'):
    !git clone --recursive https://github.com/camenduru/gaussian-splatting
else:
    print("仓库已存在，跳过克隆。")

!pip install -q plyfile
print("plyfile 安装完毕。")

# --- 新增步骤 3.5: 修复 simple-knn 源代码 ---
print("\n--- 3.5 修复 simple-knn 源代码 (添加 missing include) ---")
# 使用 sed -i 在 simple_knn.cu 文件的第一行插入 #include <cfloat>
# 这解决了 FLT_MAX 未定义的 C++ 编译错误
!sed -i '1i#include <cfloat>' /content/gaussian-splatting/submodules/simple-knn/simple_knn.cu
print("simple_knn.cu 源代码修复完毕。")


# --- 步骤 4: 编译 C++/CUDA 扩展 (关键步骤) ---
print("\n--- 4. 编译 C++/CUDA 扩展 ---")
%cd /content/gaussian-splatting

# 移除 -q 标志，并添加 --no-build-isolation
print("--- 正在安装 diff-gaussian-rasterization (使用 --no-build-isolation)... ---")
!pip install --no-build-isolation ./submodules/diff-gaussian-rasterization
print("--- diff-gaussian-rasterization 安装完毕。 ---")

# 同样移除 -q 标志，并添加 --no-build-isolation
# *** 我们保留 -v (verbose) 标志，以防万一 ***
print("--- 正在安装 simple-knn (使用 --no-build-isolation 和 -v)... ---")
!pip install --no-build-isolation -v ./submodules/simple-knn
print("--- simple-knn 安装完毕。 ---")

# --- 新增步骤 4.5: 验证 C++ 扩展是否安装成功 ---
print("\n--- 4.5 验证 C++ 扩展是否安装成功 ---")
try:
    # 尝试导入这两个模块的核心组件
    import diff_gaussian_rasterization
    from simple_knn._C import distCUDA2
    print("CUDA 扩展模块 (diff_gaussian_rasterization, simple_knn) 导入成功！")
    print("编译和安装已确认成功。")
except ImportError as e:
    print(f"\n--- 验证失败 ---")
    print(f"错误：模块导入失败: {e}")
    print("这表明 C++/CUDA 扩展的编译和安装最终还是失败了。")
    print("请仔细检查上面步骤 4 中的编译日志，查找 'error:' 相关的具体 C++ 或 CUDA 错误信息。")
    sys.exit("因 C++ 扩展安装失败，停止执行。")

# --- 步骤 5: 下载数据并开始训练 ---
print("\n--- 5. 下载数据并开始训练 ---")
if not os.path.exists('tandt_db.zip'):
    !wget https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip
else:
    print("数据文件已存在，跳过下载。")

if not os.path.exists('tandt'):
    !unzip -q tandt_db.zip # 解压用 -q 保持安静
else:
    print("数据目录已存在，跳过解压。")

print("\n--- 开始训练 (train.py) ---")
# 开始训练
!python train.py -s /content/gaussian-splatting/tandt/train

--- 1. 检查环境 ---
GPU 检测成功: Tesla T4
PyTorch CUDA 版本: 12.6
环境检查通过。

--- 2. 更新 pip 和构建工具 ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 82.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
构建工具更新完毕。

--- 3. 克隆仓库并安装 plyfile ---
/content
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 603, done.
remote: Total 603 (delta 0), reused 0 (delta 0), pack-reused 603 (from 1)
Receiving objects: 100% (603/603), 2.09 MiB | 5.67 MiB/s, done.
Resolving deltas: 100% (344/344), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization) re

In [2]:
# --- 新增步骤 6: 渲染结果 (生成视频) ---
print("\n--- 步骤 6: 渲染结果 (生成视频) ---")
print("训练完成。正在查找最新的模型路径...")

# 自动查找最新的模型文件夹 (output/ 目录中按时间戳最新的那个)
# 我们使用 ls -td ... | head -n 1 来获取最新的目录
# 注意：!export 在 Colab 的 shell 命令间不持久，所以我们用分号在同一行执行
!MODEL_PATH=$(ls -td /content/gaussian-splatting/output/*/ | head -n 1); \
 echo "找到模型路径: $MODEL_PATH"; \
 \
 echo "--- 正在运行 render.py ... ---"; \
 python render.py -m $MODEL_PATH; \
 \
 echo "--- 正在使用 ffmpeg 合成视频... ---"; \
 ffmpeg -framerate 10 -i $MODEL_PATH/train/ours_30000/renders/%05d.png -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" -c:v libx264 -r 10 -pix_fmt yuv420p /content/render_video.mp4 -y -loglevel error; \
 \
 echo "--- 视频合成完毕! ---"; \
 echo "请在左侧文件栏中下载 /content/render_video.mp4 查看结果。"


--- 步骤 6: 渲染结果 (生成视频) ---
训练完成。正在查找最新的模型路径...
找到模型路径: /content/gaussian-splatting/output/4ac49d6f-2/
--- 正在运行 render.py ... ---
Looking for config file in /content/gaussian-splatting/output/4ac49d6f-2/cfg_args
Config file found: /content/gaussian-splatting/output/4ac49d6f-2/cfg_args
Rendering /content/gaussian-splatting/output/4ac49d6f-2/
Loading trained model at iteration 30000 [11/11 14:39:41]
Reading camera 301/301 [11/11 14:39:43]
Loading Training Cameras [11/11 14:39:43]
Loading Test Cameras [11/11 14:39:48]
Rendering progress: 100% 301/301 [02:15<00:00,  2.23it/s]
Rendering progress: 0it [00:00, ?it/s]
--- 正在使用 ffmpeg 合成视频... ---
--- 视频合成完毕! ---
请在左侧文件栏中下载 /content/render_video.mp4 查看结果。


In [4]:
# --- 新增步骤 6: 渲染结果 (生成视频) ---
print("\n--- 步骤 6: 渲染结果 (生成视频) ---")
print("训练完成。正在查找最新的模型路径...")

# 自动查找最新的模型文件夹 (output/ 目录中按时间戳最新的那个)
# 我们使用 ls -td ... | head -n 1 来获取最新的目录
# 注意：!export 在 Colab 的 shell 命令间不持久，所以我们用分号在同一行执行
!MODEL_PATH=$(ls -td /content/gaussian-splatting/output/*/ | head -n 1); \
 echo "找到模型路径: $MODEL_PATH"; \
 \
 echo "--- 正在运行 render.py ... ---"; \
 python render.py -m $MODEL_PATH; \
 \
 echo "--- 正在使用 ffmpeg 合成视频... ---"; \
 ffmpeg -framerate 10 -i $MODEL_PATH/train/ours_30000/gt/%05d.png -vf "pad=ceil(iw/2)*2:ceil(ih/2)*2" -c:v libx264 -r 10 -pix_fmt yuv420p /content/render_gt_video.mp4 -y -loglevel error; \
 \
 echo "--- 视频合成完毕! ---"; \
 echo "请在左侧文件栏中下载 /content/render_gt_video.mp4 查看结果。"


--- 步骤 6: 渲染结果 (生成视频) ---
训练完成。正在查找最新的模型路径...
找到模型路径: /content/gaussian-splatting/output/4ac49d6f-2/
--- 正在运行 render.py ... ---
Looking for config file in /content/gaussian-splatting/output/4ac49d6f-2/cfg_args
Config file found: /content/gaussian-splatting/output/4ac49d6f-2/cfg_args
Rendering /content/gaussian-splatting/output/4ac49d6f-2/
Loading trained model at iteration 30000 [11/11 14:57:42]
Reading camera 301/301 [11/11 14:57:44]
Loading Training Cameras [11/11 14:57:44]
Loading Test Cameras [11/11 14:57:49]
Rendering progress: 100% 301/301 [02:14<00:00,  2.23it/s]
Rendering progress: 0it [00:00, ?it/s]
--- 正在使用 ffmpeg 合成视频... ---
--- 视频合成完毕! ---
请在左侧文件栏中下载 /content/render_gt_video.mp4 查看结果。
